In [3]:
# ==============================================================================
# CELL 1: Environment Setup & Fast CUDA Wheel Installation (<10 seconds)
# ==============================================================================
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Install pre-compiled CUDA wheel instead of compiling from source
print("🚀 Installing pre-built CUDA wheel for llama-cpp-python...")
!pip install --no-cache-dir llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# 3. Install Flask & PyNgrok dependencies
!pip install -q flask flask-cors pyngrok requests

# 4. Verify CUDA acceleration is enabled
from llama_cpp import llama_supports_gpu_offload
print(f"\n✅ Installation complete! GPU Offload Supported: {llama_supports_gpu_offload()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Installing pre-built CUDA wheel for llama-cpp-python...
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 168.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 7.2 MB/s eta 0:00:00


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB



✅ Installation complete! GPU Offload Supported: True


In [ ]:
# ==============================================================================
# CELL 2: Load GGUF Model onto GPU & Launch Flask Server
# ==============================================================================
import os, gc
import torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata
from llama_cpp import Llama

# 1. Paths & Authentication
MODEL_PATH = "/content/drive/MyDrive/medikiosk-AyurParam/model/ayurparam-q4_k_m.gguf"
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# Verify model file existence
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Model file not found at: {MODEL_PATH}")

# 2. Load GGUF Model onto GPU
# n_gpu_layers=-1 offloads ALL model layers to CUDA VRAM for maximum speed
print(f"🚀 Loading AyurParam GGUF model from {MODEL_PATH}...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,      # Offload all layers to GPU
    n_ctx=4096,           # Context window length
    n_batch=512,          # Batch size for prompt evaluation
    verbose=False
)
print("✅ AyurParam GGUF loaded successfully onto GPU!")

# 3. Initialize Flask App
app = Flask(__name__)
CORS(app)

@app.after_request
def add_ngrok_headers(response):
    response.headers["ngrok-skip-browser-warning"] = "true"
    return response

@app.route("/", methods=["GET"])
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "service": "MediKiosk AyurParam GGUF Microservice",
        "status": "ok",
        "model_loaded": llm is not None,
        "format": "GGUF Q4_K_M (CUDA Accelerated)",
        "model_path": MODEL_PATH
    })

@app.route("/gpu-status", methods=["GET"])
def gpu_status():
    if not torch.cuda.is_available():
        return jsonify({"cuda_available": False})

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    return jsonify({
        "cuda_available": True,
        "allocated_vram_mb": round(allocated, 2),
        "reserved_vram_mb": round(reserved, 2)
    })

@app.route("/generate", methods=["POST"])
@app.route("/api/generate", methods=["POST"])
def generate():
    global llm
    if llm is None:
        return jsonify({"error": "Model is currently unloaded."}), 500

    try:
        data = request.get_json(force=True) or {}
        prompt = data.get("prompt") or data.get("inputs") or data.get("text") or ""
        system_prompt = data.get("system_prompt", "You are AyurParam, an expert Ayurvedic clinical assistant specializing in Prakriti assessment, Vikriti diagnosis, and holistic healthcare.")
        max_tokens = int(data.get("max_tokens", 512))
        temperature = float(data.get("temperature", 0.2))

        if not prompt:
            return jsonify({"error": "Missing 'prompt' field"}), 400

        # Construct Chat Completion format
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        output = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9
        )

        response_text = output["choices"][0]["message"]["content"].strip()
        tokens_generated = output["usage"]["completion_tokens"]

        return jsonify({
            "response": response_text,
            "tokens_generated": tokens_generated,
            "status": "success",
            "model": "ayurparam-q4_k_m.gguf"
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

    finally:
        # VRAM Cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

@app.route("/unload", methods=["POST"])
def unload():
    global llm
    llm = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 VRAM cleared. AyurParam model unloaded.")
    return jsonify({"status": "success", "message": "AyurParam model evicted from GPU memory."})

# 4. Start PyNgrok Tunnel
ngrok.kill()
tunnel = ngrok.connect(5000, "http")
print(f"\n==================================================")
print(f"🔗 AYURPARAM_API_URL = {tunnel.public_url}")
print(f"==================================================\n")

app.run(host="0.0.0.0", port=5000)

🚀 Loading AyurParam GGUF model from /content/drive/MyDrive/medikiosk-AyurParam/model/ayurparam-q4_k_m.gguf...


llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


✅ AyurParam GGUF loaded successfully onto GPU!

🔗 AYURPARAM_API_URL = https://doormat-undying-detergent.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:18:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:18:54] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:19:47] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:20:00] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:20:03] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:20:21] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:20:32] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:20:34] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 03:25:39] "GET /health H